# EMA Support/Resistance Scanner

## Tool for Ranking Moving Averages by Market Interaction

## Table of contents

- [Core pipeline: Fetch → Analyze → Run functions](#Core-pipeline:-Fetch-→-Analyze-→-Run-functions)
- [Exploratory EMA Statistics](#Exploratory-EMA-Statistics)
- [Support & Resistance Analysis](#Support-&-Resistance-Analysis)
  - [Filters](#Filters)
  - [EMA as Support](#EMA-as-Support)
  - [EMA as Resistance](#EMA-as-Resistance)
  - [Universal EMA](#Universal-EMA)
- [Visual analysis](#Visual-analysis)
- [Backtesting](#Backtesting)
- [Data export](#Data-export)
- [Conclusion](#Conclusion)


__Goals:__
1. Build a ByBit Perpetual Futures EMA S/R Analyzer.
2. Quantify and compare EMA behavior to understand which periods price actually respects.

__Data:__ \
ByBit Perpetual Futures.

__Workflow:__ \
For each EMA period in a user-supplied range, every evaluated candle is placed in exactly one of five mutually-exclusive categories, so the categories always sum to evaluated_candles.

__Inputs:__
- symbol
- interval (ByBit codes: 1/3/5/15/30/60/120/240/360/720/D/W/M)
- start/end dates (YYYY-MM-DD)
- ema_range (any iterable of integers, e.g. range(1, 101) or [9, 21, 50, 200])
- delta
- delta_mode

__Delta modes:__
- "percent" — delta is a % of the EMA value at each candle (recommended, scales across price regimes) \
The allowed distance changes with price.
- "absolute" — delta is a fixed price distance in quote currency (e.g. USDT). Distance in price units. \
The allowed distance is always exactly the chosen 'number' regardless of price.

__Categories__ (mutually exclusive partition; every candle lands in one category):
- cross      → Low ≤ EMA ≤ High. The candle's range straddles the EMA — the level was *broken*, not held.
- low_touch  → Low > EMA AND Low − EMA ≤ delta. Entire candle above EMA; low approached EMA from above without crossing — a *test of support*.
- high_touch → High < EMA AND EMA − High ≤ delta. Entire candle below EMA; high approached EMA from below without crossing — a *test of resistance*.
- above      → Low > EMA AND Low − EMA > delta. Entire candle above EMA, low far away — clean uptrend, no test.
- below      → High < EMA AND EMA − High > delta. Entire candle below EMA, high far away — clean downtrend, no test.

__Invariants:__
- cross + low_touch + high_touch + above + below = evaluated_candles
- any_touch = low_touch + high_touch (the two are mutually exclusive — a candle either tests support OR resistance, never both)

__Notes:__
- The first period candles are skipped per EMA (warm-up); toggle with skip_warmup=False if you don't want that.
- Pagination handles arbitrarily long date ranges (1000-candle chunks).
- Uses ewm(span=period, adjust=False) — standard Wilder/TradingView-style EMA.


## Core pipeline: Fetch → Analyze → Run functions

In [25]:
import time
import requests
import pandas as pd
import numpy as np
from typing import Iterable
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [26]:
BYBIT_API = "https://api.bybit.com/v5/market/kline"

__1. Data fetch__

In [27]:
def fetch_bybit_klines(
    symbol: str,
    interval: str,
    start: str,
    end: str,
    category: str = "linear",       # perpetual futures (USDT-margined)
) -> pd.DataFrame:
    """
    Pull kline data from ByBit v5 with automatic pagination.

    symbol   : e.g. "BTCUSDT"
    interval : "1","3","5","15","30","60","120","240","360","720","D","W","M"
    start/end: ISO date strings, e.g. "2024-01-01"
    category : "linear"  -> USDT perps (default)
               "inverse" -> coin-margined perps
    """
    # Convert dates to milliseconds
    # end date is exclusive
    # to extend to end-of-day (so include the end date): end_ts = int(pd.Timestamp(end, tz="UTC").timestamp() * 1000) + 86_400_000 - 1
    start_ts = int(pd.Timestamp(start, tz="UTC").timestamp() * 1000)
    end_ts   = int(pd.Timestamp(end,   tz="UTC").timestamp() * 1000)

    rows, cursor_end = [], end_ts

    while True:
        params = {
            "category": category,
            "symbol":   symbol.upper(),
            "interval": interval,
            "start":    start_ts,
            "end":      cursor_end,
            "limit":    1000,                 # ByBit max per request
        }
        r = requests.get(BYBIT_API, params=params, timeout=20)
        r.raise_for_status()
        data = r.json()

        if data.get("retCode") != 0:
            raise RuntimeError(f"ByBit API error: {data}")

        batch = data["result"]["list"]        # newest first
        if not batch:
            break

        rows.extend(batch)

        # Result_list is sorted DESCENDING (newest candle first)
        oldest_ts = int(batch[-1][0])
        if oldest_ts <= start_ts or len(batch) < 1000:
            break
        # Move backward to fetch older candles
        cursor_end = oldest_ts - 1
        time.sleep(0.1)                       # polite rate-limiting

    if not rows:
        raise ValueError(
            f"No candles returned for {symbol} {interval} between {start} and {end}"
        )

    # Create DataFrame
    cols = ["timestamp", "open", "high", "low", "close", "volume", "turnover"]
    df = pd.DataFrame(rows, columns=cols)

    # Convert types
    df = df.astype({c: float for c in cols[1:]})
    df["timestamp"] = pd.to_datetime(df["timestamp"].astype(np.int64), unit="ms", utc=True)
    
    # Remove any duplicates and sort ascending by time
    df = (df.drop_duplicates("timestamp")
            .sort_values("timestamp")
            .reset_index(drop=True))
    
    # Filter exact date range (inclusive)
    df = df[(df["timestamp"] >= pd.Timestamp(start, tz="UTC")) &
            (df["timestamp"] <= pd.Timestamp(end,   tz="UTC"))].reset_index(drop=True)
    
    return df


__2. Touch / cross analysis__

Main algorithm:
1. Fetches ByBit perpetual futures data.
2. For every EMA period in ema_range:
   - Computes the EMA on close prices.
   - Assigns every evaluated candle to a category.
   - Categories are mutually exclusive.
3. Returns a clean DataFrame with one row per EMA period.


Partition table: every evaluated candle is placed in exactly one category.

| category     | condition                                                            | behavior | meaning |
|--------------|----------------------------------------------------------------------|---------|----------|
| cross      | Low ≤ EMA ≤ High | candle crosses EMA | level broken             |
| low_touch  | Low > EMA AND Low − EMA ≤ delta | entire candle above EMA, Low within delta of EMA, Low approached EMA from above | support held, no break |
| high_touch | High < EMA AND EMA − High ≤ delta | entire candle below EMA, High within delta of EMA, High approached EMA from below | resistance held, no break |
| above      | Low > EMA AND Low − EMA > delta | entire candle above EMA, Low further than delta | clean uptrend, no test of EMA |
| below      | High < EMA AND EMA − High > delta | entire candle below EMA, High further than delta | clean downtrend, no test of EMA |

Invariants:
- cross + low_touch + high_touch + above + below = evaluated_candles
- any_touch = low_touch + high_touch (mutually exclusive — a candle either tests support OR resistance, never both)


In [28]:
# skip_warmup=True: ignore the first N candles (where N = the EMA period)
# Example: For EMA 50, it throws away the first 50 candles and only counts touches/cross starting from candle 51.
# Set it to False if you want to include every candle regardless.

# delta %: the allowed distance changes with price
# delta absolute: the allowed distance is always exactly the chosen 'number' regardless of price

def analyze_ema_touches(
    df: pd.DataFrame,
    ema_range: Iterable[int],
    delta: float,
    delta_mode: str = "percent",   # "percent" (of EMA) or "absolute"
    skip_warmup: bool = True,
) -> pd.DataFrame:
    """
    Per EMA period, every evaluated candle is placed in exactly ONE of these
    mutually exclusive buckets:

        cross      : Low <= EMA <= High           (range straddles the EMA)
        low_touch  : Low > EMA  AND  Low - EMA  <= delta
                     (entire candle above EMA, low close enough to register a touch)
        high_touch : High < EMA  AND  EMA - High <= delta
                     (entire candle below EMA, high close enough to register a touch)
        above      : Low > EMA  AND  Low - EMA  > delta
                     (entire candle above EMA, low far from EMA = clean trend)
        below      : High < EMA  AND  EMA - High > delta
                     (entire candle below EMA, high far from EMA = clean trend)

    Quality counters (used by the rejection-rate ratios downstream).

    A cross's direction is resolved by the bar's `open` relative to EMA:
        cross_from_above = cross AND open > EMA   # bar started above; range pierced down
        cross_from_below = cross AND open < EMA   # bar started below; range pierced up
    A tie (open == EMA exactly) counts as neither — vanishingly rare with float prices.

        support_test    = low_touch  + cross_from_above                          # approaches from above
        support_held    = low_touch  + (cross_from_above AND close > EMA)        # ... that ended strictly above
        resistance_test = high_touch + cross_from_below                          # approaches from below
        resistance_held = high_touch + (cross_from_below AND close < EMA)        # ... that ended strictly below

    Invariants that always hold:
        cross + low_touch + high_touch + above + below = evaluated_candles
        any_touch        = low_touch + high_touch       (no double counting)
        cross_from_above + cross_from_below + cross_at_open_tie = cross
        support_held    <= support_test
        resistance_held <= resistance_test
    """
    close, high, low, open_ = df["close"], df["high"], df["low"], df["open"]
    results = []

    for period in ema_range:
        # Compute EMA (pandas built-in, adjust=False = classic EMA)
        ema = close.ewm(span=period, adjust=False).mean()

        if delta_mode == "percent":
            tol = ema.abs() * (delta / 100.0)
        elif delta_mode == "absolute":
            tol = pd.Series(delta, index=ema.index)
        else:
            raise ValueError("delta_mode must be 'percent' or 'absolute'")

        # Optionally skip the first `period` candles while the EMA is warming up
        warmup = period if skip_warmup else 0
        mask = pd.Series(False, index=ema.index)
        mask.iloc[warmup:] = True

        # Top-level partition: cross / strictly-above / strictly-below
        crossed         = (low <= ema) & (ema <= high) & mask
        strictly_above  = (low > ema) & mask
        strictly_below  = (high < ema) & mask

        # Touches are *near approaches that did not cross* — split out from
        # strictly_above / strictly_below by the delta criterion.
        low_touch  = strictly_above & ((low  - ema) <= tol)
        high_touch = strictly_below & ((ema - high) <= tol)
        any_touch  = low_touch | high_touch

        # 'above' / 'below' = clean trending candles (strictly past EMA, NOT a touch)
        above = strictly_above & ~low_touch
        below = strictly_below & ~high_touch

        # Resolve cross direction by the bar's OPEN relative to EMA.
        # A cross opened above EMA is a support test (price came down from above).
        # A cross opened below EMA is a resistance test (price came up from below).
        # Strict inequalities: a tie (open == EMA, or close == EMA on the held check)
        # counts as neither — avoids the EMA1 degeneracy where close == EMA always.
        crossed_from_above = crossed & (open_ > ema)
        crossed_from_below = crossed & (open_ < ema)
        crossed_held_above = crossed_from_above & (close > ema)   # opened above, closed back above → support held
        crossed_held_below = crossed_from_below & (close < ema)   # opened below, closed back below → resistance held

        # Rejection-rate counters. Each cross is now counted in exactly one direction
        # (or neither, for ties). low_touch is strictly_above by definition, so it
        # automatically has open > EMA and close > EMA — always a "support held" event.
        support_test    = low_touch  | crossed_from_above
        support_held    = low_touch  | crossed_held_above
        resistance_test = high_touch | crossed_from_below
        resistance_held = high_touch | crossed_held_below

        results.append({
            "ema":               period,
            "low_touch":         int(low_touch.sum()),
            "high_touch":        int(high_touch.sum()),
            "any_touch":         int(any_touch.sum()),
            "above":             int(above.sum()),
            "below":             int(below.sum()),
            "cross":             int(crossed.sum()),
            "cross_above":       int(crossed_from_above.sum()),
            "cross_below":       int(crossed_from_below.sum()),
            "support_test":      int(support_test.sum()),
            "support_held":      int(support_held.sum()),
            "resistance_test":   int(resistance_test.sum()),
            "resistance_held":   int(resistance_held.sum()),
            "evaluated_candles": int(mask.sum()),
        })

    return pd.DataFrame(results)


__3. Calling the fetch and analyze functions__

Configuration:
- interval = "60" for 1-hour candles
- ema_range  = range(5, 201, 5) calculates EMA 5, 10, 15, ..., 200
- delta_mode = "percent" or "absolute"
- delta %: the allowed distance between EMA and candle changes with price
- delta absolute: the allowed distance is always exactly the chosen 'number' regardless of price. Distance in price units.

*Examples:*
- delta=0.10 is 0.10 % of the EMA 
- a 5 USDT price difference in % is calculated as:  5 / price (e.g. 67,000) × 100 = 0.00746%
- a 0.0075% delta in USDT is calculated as: price (e.g. 67,000) × 0.0075 / 100 = 5.025 USDT

In [29]:
symbol     = "BTCUSDT"
interval   = "15"
start      = "2026-01-01"
end        = "2026-04-01"
ema_range  = range(1, 200, 1)
delta      = 20
delta_mode = "absolute"

Dataframe of raw OHLCV candles (klines) from ByBit:

In [30]:
df = fetch_bybit_klines(symbol, interval, start, end)

print(f"Fetching {symbol} interval={interval} from {start} to {end} ...")
print(f"  -> {len(df)} candles downloaded.")

Fetching BTCUSDT interval=15 from 2026-01-01 to 2026-04-01 ...
  -> 8641 candles downloaded.


Dataframe of candle categories by EMA period:

In [31]:
result = analyze_ema_touches(df, ema_range, delta, delta_mode)

result.head(10).style.hide(axis="index")

ema,low_touch,high_touch,any_touch,above,below,cross,cross_above,cross_below,support_test,support_held,resistance_test,resistance_held,evaluated_candles
1,0,0,0,0,0,8640,4312,4325,4312,0,4325,0,8640
2,30,33,63,3,3,8570,4289,4281,4319,756,4314,718,8639
3,159,130,289,110,103,8136,4086,4050,4245,1354,4180,1281,8638
4,289,239,528,362,351,7396,3722,3674,4011,1658,3913,1588,8637
5,369,346,715,673,633,6615,3327,3288,3696,1724,3634,1728,8636
6,375,405,780,997,943,5915,2983,2932,3358,1688,3337,1724,8635
7,433,426,859,1231,1241,5303,2660,2643,3093,1621,3069,1643,8634
8,415,399,814,1470,1502,4847,2411,2436,2826,1513,2835,1519,8633
9,399,393,792,1648,1684,4508,2248,2260,2647,1439,2653,1467,8632
10,411,383,794,1811,1858,4168,2072,2096,2483,1356,2479,1389,8631


__Alternative: convenience wrapper__

In [ ]:
# def run(
#     symbol: str,
#     interval: str,
#     start: str,
#     end: str,
#     ema_range: Iterable[int],
#     delta: float,
#     delta_mode: str = "percent",
# ) -> tuple[pd.DataFrame, pd.DataFrame]:
#     print(f"Fetching {symbol} interval={interval} from {start} to {end} ...")
    
#     df = fetch_bybit_klines(symbol, interval, start, end)
#     print(f"  -> {len(df)} candles downloaded")

#     result = analyze_ema_touches(df, ema_range, delta, delta_mode)
#     return df, result

In [ ]:
# symbol     = "BTCUSDT"
# interval   = "15"
# start      = "2026-01-01"
# end        = "2026-04-01"
# ema_range  = range(1, 200, 1)
# delta      = 40
# delta_mode = "absolute"

In [ ]:
# df, result = run(symbol, interval, start, end, ema_range, delta, delta_mode)

## Exploratory EMA Statistics

__Distributions of EMA periods across candle categories.__

EMA periods are grouped by their count for each candle category.

For each candle category, the histogram shows how the count is distributed across the EMA range — i.e. how many EMAs had ~100 touches, ~500, ~2000, etc.

Use it to see the bulk vs. the tails: where most EMAs cluster (typical behavior) and which ones sit far from the pack (outliers worth a closer look).

Chart explanation:
- each chart counts EMA periods, not candles
- the x-axis is "number of times this candle behavior happened"
- the y-axis is "how many EMAs had that number"
- a tall bar in the middle → most EMAs behave this way. That's the typical range.
- short bars far on the sides → a few EMAs whose counts are unusually high or low. Those are the outliers worth a closer look.


For example, the low_touch histogram:
- Answers "how many of the 199 EMAs had around 100 low_touches? around 300? around 400?"
- Each of the 199 EMAs has a low_touch count (e.g. EMA-1 → 0, EMA-50 → 289, EMA-200 → 18).
- The histogram bins those 199 numbers and shows how many EMAs fall into each bin.
- So if 12 EMAs had between 240 and 260 low_touches, the bar centered at 250 has height 12.

In [ ]:
result[["low_touch", "high_touch", "any_touch", "cross", "above", "below"]].hist(bins=100, figsize=(12, 8)) 

__Which EMA periods are touched most often by candle Lows?__

In [ ]:
emas_sorted_by_lows = (result[["ema", "low_touch"]]
           .sort_values("low_touch", ascending=False)
           .set_index("ema"))

emas_sorted_by_lows

__Which EMA periods are touched most often by candle Highs?__

In [ ]:
emas_sorted_by_highs = (result[["ema", "high_touch"]]
           .sort_values("high_touch", ascending=False)
           .set_index("ema"))

emas_sorted_by_highs

__Which EMA periods have the most candles floating entirely above them?__

Price spent most of the period trending above this EMA (EMA acted as a floor / bullish regime).

In [ ]:
emas_sorted_by_above = (result[["ema", "above"]]
           .sort_values("above", ascending=False)
           .set_index("ema"))

emas_sorted_by_above

__Which EMA periods have the most candles floating entirely below them?__

Price spent most of the period trending below this EMA (EMA acted as a ceiling / bearish regime).

In [ ]:
emas_sorted_by_below = (result[["ema", "below"]]
           .sort_values("below", ascending=False)
           .set_index("ema"))

emas_sorted_by_below

## Support & Resistance Analysis

### Filters

2 filters for opposite EMA extremes:
- cross frequency: EMAs price crosses too often
- sample size: EMAs price tests too rarely

What's left in the middle is the tradeable zone — EMAs that price interacts with often enough to evaluate, but not so often that the EMA hugs price.

__*1. Cross-frequency filter*__

Filters out EMAs that are crossed too frequently.

"Is this EMA structurally able to act as S/R at all?"

If price crosses an EMA on most bars, the EMA isn't a wall — it's tracking price. \
Very fast EMAs (short periods) stick too closely to price to act as meaningful support or resistance. \
Price crosses them on almost every candle, so conditions like “close ≥ EMA” or “close ≤ EMA” become trivial and misleading.

At the extreme, an EMA with period = 1 (EMA(span=1) == close) is identical to the close price. \
This makes the support ratio artificially equal to 1.0—even though the EMA provides no real support.

Any EMA whose cross rate (cross / evaluated_candles) exceeds MAX_CROSS_RATE is excluded from all ratio calculations. \
These EMAs aren’t true support/resistance levels — they’re just noise.

The MAX_CROSS_RATE threshold is defined once and reused consistently:
- 0.5 → keeps EMAs crossed in less than 50% of candles
- 0.3 → keeps EMAs crossed in less than 30% of candles
- 0.2 → keeps only slower EMAs that behave as strong, decisive S/R levels

Lower threshold = stricter filter = keeps only EMAs that price visits less often = fewer, more meaningful EMAs.

__*2. Sample-size filter*__

Ensures the S/R rate is supported by enough observations to be meaningful:
- each EMA is evaluated on a sufficient number of tests
- prevents evaluation on too few observations

"Do we have enough data to trust the ratio?"

A 100% hold rate over 3 tests is statistical noise. With such a small sample, the result isn’t reliable. \
A 70% hold rate over 200 tests is a real signal. 

Rule of thumb:
- Require a minimum number of touches that makes sense for your dataset.
- A simple guideline is at least ~1 touch per week of data.

Example: \
8640 candles of 15-minute data ≈ 90 days. \
Setting MIN_TOUCHES = 30 means ~1 touch every 3 days → a reasonable minimum.

Ways to set the threshold:
- Fixed threshold:
    - Choose a number that feels statistically meaningful: \
      MIN_TOUCHES = int
- Data-driven threshold (auto-scale):
    - Keep EMAs that have more touches than the median: \
      MIN_TOUCHES = int(result["support_test"].median())
    - Be more selective using a quantile: \
      MIN_TOUCHES = int(result["support_test"].quantile(0.66)) \
      (keeps only the most frequently tested EMAs: top ~33%)


The MIN_TOUCHES threshold can be set separately for support, resistance, and universal EMAs.

More touches = more reliable evaluation.

__Filters configuration__

In [32]:
# Cross-frequency filter:
# drop EMAs whose cross rate exceeds this threshold
# require the cross rate to stay below this threshold for the ratio to be meaningful

MAX_CROSS_RATE = 0.3

def filter_by_cross_rate(df):
    """Add a cross_rate column and keep only rows below MAX_CROSS_RATE.

    The input df must include `cross` and `evaluated_candles` columns.
    """
    df = df.copy()
    df["cross_rate"] = df["cross"] / df["evaluated_candles"]
    return df[df["cross_rate"] < MAX_CROSS_RATE]


# Sample-size filters (per direction):
# drop EMAs whose test count falls below this threshold
# require at least this many tests for the ratio to be meaningful

# MIN_TOUCHES_SUP = int(result["support_test"].median())
# MIN_TOUCHES_RES = int(result["resistance_test"].median())
# MIN_TOUCHES_UNI = int((result["any_touch"] + result["cross"]).median())
MIN_TOUCHES_SUP = 30
MIN_TOUCHES_RES = 30
MIN_TOUCHES_UNI = 60

### EMA as Support

High ratio = EMA acts as support: price dips down to it, touches or briefly crosses, but the close finishes back above without committing to a breakdown.

Adjacent EMAs clustering with nearly identical ratios: is the signature of a real support zone.

Which formula to use?

| # | formula | meaning | what it measures | usage |
|---|---|---|---|---|
| 1 | support_held / support_test | hold rate: of all support tests, share that held | Which EMA is structurally the strongest support? When this EMA is tested from above (touched or crossed), how often does it hold (closed back above it)? | support-quality metric |
| 2 | low_touch / support_test | rejection rate: of all support tests, share that rejected cleanly | How clean is this EMA's support (pure wick rejections, any pierce is a failure even if it recovered)? | stricter support-quality metric |
| 3 | low_touch / evaluated_candles | frequency of clean support tests | Which EMA produces the most tradeable support setups in this dataset? How often does price reach this EMA? | tradability, activity gauge |
| 4 | (low_touch + above) / cross | bullishness at this EMA | For every pierce, how many bars stayed entirely above? How dominant is the "above" side relative to break-throughs? | regime / trend-bias ratio |
| 5 | (low_touch + above) / evaluated_candles | regime indicator | Fraction of bars where the entire candle was above the EMA. | trend filter |


__*1. Hold rate: is this EMA a good support level to trade?*__

EMA as a support quality metric: is this EMA a reliable bounce level?

ratio_support_1 shows the % of support tests that held:
- 0.0 → every approach broke through (no holds)
- 0.5 → bounces half the time when tested
- 1.0 → every approach held (perfect support)

A support test = the bar approached the EMA from above (low_touch, or a cross that opened above the EMA). \
A test held = the bar didn't end below the EMA (close > EMA). A close exactly at the EMA counts as neither held nor broken.

In [33]:
# The unfiltered table
# replace(0, np.nan) handles division by zero (an EMA may have no support tests at all)
# .copy() avoids SettingWithCopyWarning and keeps modifications local to ratio_support_1

ratio_support_1 = result[["ema", "support_test", "support_held", "cross", "evaluated_candles"]].copy()
ratio_support_1["ratio_support_1"] = ratio_support_1["support_held"] / ratio_support_1["support_test"].replace(0, np.nan)
ratio_support_1 = ratio_support_1.sort_values("ratio_support_1", ascending=False).set_index("ema")

ratio_support_1


,support_test,support_held,cross,evaluated_candles,ratio_support_1
ema,,,,,
114,591,369,920,8527,0.624365
115,588,366,913,8526,0.622449
116,574,356,908,8525,0.620209
117,574,355,912,8524,0.618467
118,576,356,909,8523,0.618056
...,...,...,...,...,...
5,3696,1724,6615,8636,0.466450
4,4011,1658,7396,8637,0.413363
3,4245,1354,8136,8638,0.318963


*Conclusion:*
- Slow EMAs (above ~100) often score highest because price rarely reaches them — when they are tested, the test tends to be a wick rejection rather than a clean breakdown.
- The ratio ceiling tells you how well the best EMA contained price for the period studied.
- Always weight the ranking by sample size (support_test).
- Re-run on different regimes to see how stable each EMA's hold rate is.


__Filtering out the results__

In [34]:
# Top 10 EMAs that pass both filters: cross-frequency (structural) and sample-size (statistical)

ratio_support_1_filtered = filter_by_cross_rate(ratio_support_1)
ratio_support_1_filtered = ratio_support_1_filtered[ratio_support_1_filtered["support_test"] >= MIN_TOUCHES_SUP]
ratio_support_1_filtered.head(10)


,support_test,support_held,cross,evaluated_candles,ratio_support_1,cross_rate
ema,,,,,,
114,591,369,920,8527,0.624365,0.107893
115,588,366,913,8526,0.622449,0.107084
116,574,356,908,8525,0.620209,0.106510
117,574,355,912,8524,0.618467,0.106992
118,576,356,909,8523,0.618056,0.106653
113,583,360,921,8528,0.617496,0.107997
77,669,413,1136,8564,0.617339,0.132648
112,579,354,930,8529,0.611399,0.109040
119,560,342,911,8522,0.610714,0.106900


__*2. Rejection rate: how clean is this EMA's support?*__

Stricter metric: of all approaches from above, what fraction were clean wick rejections (didn't pierce)? \
Denominator: support_test = low_touch + cross_from_above by construction

In [35]:
ratio_support_2 = result[["ema", "low_touch", "support_test", "cross", "evaluated_candles"]].copy()
ratio_support_2["ratio_support_2"] = ratio_support_2["low_touch"] / ratio_support_2["support_test"].replace(0, np.nan)
ratio_support_2 = ratio_support_2.sort_values("ratio_support_2", ascending=False).set_index("ema")

ratio_support_2


,low_touch,support_test,cross,evaluated_candles,ratio_support_2
ema,,,,,
114,125,591,920,8527,0.211506
113,120,583,921,8528,0.205832
115,119,588,913,8526,0.202381
116,114,574,908,8525,0.198606
71,137,706,1180,8570,0.194051
...,...,...,...,...,...
5,369,3696,6615,8636,0.099838
4,289,4011,7396,8637,0.072052
3,159,4245,8136,8638,0.037456


__Filtering out the results__

In [36]:
ratio_support_2_filtered = filter_by_cross_rate(ratio_support_2)
ratio_support_2_filtered = ratio_support_2_filtered[ratio_support_2_filtered["support_test"] >= MIN_TOUCHES_SUP]
ratio_support_2_filtered.head(10)


,low_touch,support_test,cross,evaluated_candles,ratio_support_2,cross_rate
ema,,,,,,
114,125,591,920,8527,0.211506,0.107893
113,120,583,921,8528,0.205832,0.107997
115,119,588,913,8526,0.202381,0.107084
116,114,574,908,8525,0.198606,0.106510
71,137,706,1180,8570,0.194051,0.137690
103,112,586,965,8538,0.191126,0.113024
117,109,574,912,8524,0.189895,0.106992
147,93,491,826,8494,0.189409,0.097245
51,164,871,1496,8590,0.188289,0.174156


__*3. Tradability: how often does price reach this EMA?*__

Evaluates how often the setup shows up, not how good it is when it does.

In [37]:
ratio_support_3 = result[["ema", "low_touch", "cross", "evaluated_candles"]].copy()
ratio_support_3["ratio_support_3"] = ratio_support_3["low_touch"] / ratio_support_3["evaluated_candles"].replace(0, np.nan)
ratio_support_3 = ratio_support_3.sort_values("ratio_support_3", ascending=False).set_index("ema")

ratio_support_3


,low_touch,cross,evaluated_candles,ratio_support_3
ema,,,,
7,433,5303,8634,0.050151
8,415,4847,8633,0.048071
10,411,4168,8631,0.047619
9,399,4508,8632,0.046223
11,388,3919,8630,0.044959
...,...,...,...,...
181,55,749,8460,0.006501
178,53,758,8463,0.006263
179,52,752,8462,0.006145


__*4. Bullishness: what's the trend bias around this EMA?*__

For every time price pierced the EMA (in either direction), how many bars sat entirely above it?

Captures EMAs that mark decisive, uninterrupted trend regimes. \
Bullishness amplified by stability: EMA acted as a floor and was rarely breached.

- High ratio when price spends a lot of time above with few crossings -> bullish regime.
- Rewards EMAs where price spent most time above and the EMA was rarely breached.

Example: (low_touch + above) / cross = 850 / 150 = 5.67 \
Meaning: For every pierce, ~5.7 bars sat entirely above —> strong bullish regime.

In [14]:
ratio_support_4 = result[["ema", "low_touch", "above", "cross"]].copy()
ratio_support_4["ratio_support_4"] = (ratio_support_4["low_touch"] + ratio_support_4["above"]) / ratio_support_4["cross"].replace(0, np.nan)
ratio_support_4 = ratio_support_4.sort_values("ratio_support_4", ascending=False).set_index("ema")

ratio_support_4


,low_touch,above,cross,ratio_support_4
ema,,,,
199,69,3163,674,4.795252
198,70,3163,679,4.761414
197,60,3169,688,4.693314
196,64,3170,692,4.673410
195,61,3173,697,4.639885
...,...,...,...,...
5,369,673,6615,0.157521
4,289,362,7396,0.088021
3,159,110,8136,0.033063


__*5. Trend filter: fraction of bars where the entire candle was above the EMA.*__

Captures EMAs acting as a floor in uptrends, where price often respects the level without dipping into touch distance.

- A regime indicator: high ratio means price spent the period mostly above the EMA.
- Rewards EMAs that acted as a floor during trending stretches where price never dipped close enough to register a touch.
- Useful in strong uptrends where pure touch-based ratios under-count respected levels.

In [15]:
ratio_support_5 = result[["ema", "low_touch", "above", "cross", "evaluated_candles"]].copy()
ratio_support_5["ratio_support_5"] = (ratio_support_5["low_touch"] + ratio_support_5["above"]) / ratio_support_5["evaluated_candles"].replace(0, np.nan)
ratio_support_5 = ratio_support_5.sort_values("ratio_support_5", ascending=False).set_index("ema")

ratio_support_5

,low_touch,above,cross,evaluated_candles,ratio_support_5
ema,,,,,
103,112,3352,965,8538,0.405716
113,120,3339,921,8528,0.405605
114,125,3331,920,8527,0.405301
104,103,3357,963,8537,0.405295
105,100,3357,957,8536,0.404991
...,...,...,...,...,...
5,369,673,6615,8636,0.120658
4,289,362,7396,8637,0.075373
3,159,110,8136,8638,0.031141


__Visualization: hold rate (support quality) by EMA period.__

Visualization of how often each EMA's support test held vs broke, plus the hold rate.

Lets you see raw test volume and support quality on one chart.

Result: a dual-axis chart
- stacked bars for held vs broken counts
- a red line for the hold rate

total bar height = total support tests for that EMA: out of all tests, this many held (orange) and this many broke (blue). \
broken = support_test - support_held \
hold rate = support_held / support_test

In [38]:
#Plot unfiltered EMAs
# reset_index() puts ema back as a column after using it as the index in ratio_support_1.set_index("ema")
#viz = ratio_support_1.reset_index().sort_values("ema")

#Plot only EMAs that pass both cross-frequency and sample-size filters
viz = ratio_support_1_filtered.reset_index().sort_values("ema")

ratio = viz["ratio_support_1"]
broke = viz["support_test"] - viz["support_held"]

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Stacked bars: held + broken = total support tests
fig.add_trace(
    go.Bar(x=viz["ema"], y=viz["support_held"], name="support held",
           marker_color="orange", opacity=0.75),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(x=viz["ema"], y=broke, name="support broke",
           marker_color="steelblue", opacity=0.45),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=viz["ema"], y=ratio, name="support rate",
               mode="lines", line=dict(color="red", width=2),
               hovertemplate="%{y:.1%}"),
    secondary_y=True,
)

fig.update_layout(
    title="Support held vs broken by EMA period — with hold rate",
    barmode="stack",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    hoverlabel=dict(namelength=-1),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_xaxes(title_text="EMA period", tickprefix="EMA ")
fig.update_yaxes(title_text="Count", secondary_y=False)
fig.update_yaxes(title_text="Support rate", secondary_y=True, color="red", tickformat=".0%")

fig.show()


Use it for:
- Where the ratio peaks (the red line reaches its highest points on the chart) across the EMA spectrum —> EMAs periods that acted as the best support.
- Are they isolated spikes or broad hills (a whole zone respects support)?

### EMA as Resistance

High ratio = EMA acts as resistance: price rallies up into it, the candle's high kisses or briefly crosses, and the close prints back below the EMA without committing to a breakout.

Adjacent EMAs clustering with nearly identical ratios: is the signature of a real resistance zone.

Common expectations:

__On a 15-min chart__: \
Slow EMAs like 99, 144 and 200 often score high on ratio because price rarely reaches them, and when it does it often rejects cleanly.

__In a downtrending or ranging market:__ \
The classic "respected resistance" EMAs tend to rise to the top:
- EMA 20 — short-term resistance, widely watched by intraday traders
- EMA 50 — medium-term, very popular on daily charts, often respected on lower timeframes too
- EMA 200 — long-term, the "institutional" line, strong resistance in downtrends

__In a strong uptrend:__ \
Resistance EMAs barely exist — price stays above most EMAs, so high_touch will be low across the board and the ratio becomes noisy. \
The top of ranked might be dominated by random slow EMAs with tiny sample sizes.

Which formula to use?


| # | formula | meaning | what it measures | usage |
|---|---|---|---|---|
| 1 | resistance_held / resistance_test | hold rate: of all resistance tests, share that held | Which EMA is structurally the strongest resistance? When this EMA is tested from below (touched or crossed), how often does it hold (closed back below it)? | resistance-quality metric |
| 2 | high_touch / resistance_test | rejection rate: of all resistance tests, share that rejected cleanly | How clean is this EMA's resistance (pure wick rejections, any pierce is a failure even if it recovered)? | stricter resistance-quality metric |
| 3 | high_touch / evaluated_candles | frequency of clean resistance tests | Which EMA produces the most tradeable resistance setups in this dataset? How often does price reach this EMA? | tradability, activity gauge |
| 4 | (high_touch + below) / cross | bearishness at this EMA | For every pierce, how many bars stayed entirely below? How dominant is the "below" side relative to break-throughs? | regime / trend-bias ratio |
| 5 | (high_touch + below) / evaluated_candles | regime indicator | Fraction of bars where the entire candle was below the EMA. | trend filter |


__*1. Hold rate: is this EMA a good resistance level to trade?*__


EMA as a resistance quality metric: is this EMA a reliable rejection level?

ratio_resistance_1 shows the % of resistance tests that held:
- 0.0 → every approach broke through (no holds)
- 0.5 → rejects half the time when tested
- 1.0 → every approach held (perfect resistance)

A resistance test = the bar approached the EMA from below (high_touch, or a cross that opened below the EMA). \
A test held = the bar didn't end above the EMA (close < EMA). A close exactly at the EMA counts as neither held nor broken.


In [39]:
# The unfiltered table

ratio_resistance_1 = result[["ema", "resistance_test", "resistance_held", "cross", "evaluated_candles"]].copy()
ratio_resistance_1["ratio_resistance_1"] = ratio_resistance_1["resistance_held"] / ratio_resistance_1["resistance_test"].replace(0, np.nan)
ratio_resistance_1 = ratio_resistance_1.sort_values("ratio_resistance_1", ascending=False).set_index("ema")

ratio_resistance_1


,resistance_test,resistance_held,cross,evaluated_candles,ratio_resistance_1
ema,,,,,
77,705,447,1136,8564,0.634043
76,711,449,1140,8565,0.631505
74,724,456,1153,8567,0.629834
84,669,421,1079,8557,0.629297
71,743,467,1180,8570,0.628533
...,...,...,...,...,...
5,3634,1728,6615,8636,0.475509
4,3913,1588,7396,8637,0.405827
3,4180,1281,8136,8638,0.306459


*Conclusion:*
- Ratio ceiling around 0.63 means even the best resistance EMA only rejected 63% of the time price reached it. \
The other 37% of cross, price cut through.

__Filtering out the results__

In [40]:
# Top 10 EMAs that pass both filters: cross-frequency (structural) and sample-size (statistical)

ratio_resistance_1_filtered = filter_by_cross_rate(ratio_resistance_1)
ratio_resistance_1_filtered = ratio_resistance_1_filtered[ratio_resistance_1_filtered["resistance_test"] >= MIN_TOUCHES_RES]
ratio_resistance_1_filtered.head(10)


,resistance_test,resistance_held,cross,evaluated_candles,ratio_resistance_1,cross_rate
ema,,,,,,
77,705,447,1136,8564,0.634043,0.132648
76,711,449,1140,8565,0.631505,0.133100
74,724,456,1153,8567,0.629834,0.134586
84,669,421,1079,8557,0.629297,0.126096
71,743,467,1180,8570,0.628533,0.137690
72,738,463,1170,8569,0.627371,0.136539
73,735,461,1161,8568,0.627211,0.135504
75,716,449,1149,8566,0.627095,0.134135
78,694,435,1133,8563,0.626801,0.132313


__*2. Rejection rate: how clean is this EMA's resistance?*__


Stricter metric: of all approaches from below, what fraction were clean wick rejections (didn't pierce)? \
Denominator: resistance_test = high_touch + cross_from_below by construction


In [41]:
ratio_resistance_2 = result[["ema", "high_touch", "resistance_test", "cross", "evaluated_candles"]].copy()
ratio_resistance_2["ratio_resistance_2"] = ratio_resistance_2["high_touch"] / ratio_resistance_2["resistance_test"].replace(0, np.nan)
ratio_resistance_2 = ratio_resistance_2.sort_values("ratio_resistance_2", ascending=False).set_index("ema")

ratio_resistance_2


,high_touch,resistance_test,cross,evaluated_candles,ratio_resistance_2
ema,,,,,
85,142,668,1064,8556,0.212575
100,127,612,977,8541,0.207516
138,107,516,842,8503,0.207364
82,143,693,1095,8559,0.206349
53,190,921,1430,8588,0.206298
...,...,...,...,...,...
5,346,3634,6615,8636,0.095212
4,239,3913,7396,8637,0.061078
3,130,4180,8136,8638,0.031100


__Filtering out the results__


In [42]:
ratio_resistance_2_filtered = filter_by_cross_rate(ratio_resistance_2)
ratio_resistance_2_filtered = ratio_resistance_2_filtered[ratio_resistance_2_filtered["resistance_test"] >= MIN_TOUCHES_RES]
ratio_resistance_2_filtered.head(10)


,high_touch,resistance_test,cross,evaluated_candles,ratio_resistance_2,cross_rate
ema,,,,,,
85,142,668,1064,8556,0.212575,0.124357
100,127,612,977,8541,0.207516,0.114389
138,107,516,842,8503,0.207364,0.099024
82,143,693,1095,8559,0.206349,0.127936
53,190,921,1430,8588,0.206298,0.166511
84,138,669,1079,8557,0.206278,0.126096
46,216,1050,1620,8595,0.205714,0.188482
47,210,1034,1594,8594,0.203095,0.185478
55,180,888,1392,8586,0.202703,0.162124


__*3. Tradability: how often does price reach this EMA?*__


Evaluates how often the setup shows up, not how good it is when it does.


In [43]:
ratio_resistance_3 = result[["ema", "high_touch", "cross", "evaluated_candles"]].copy()
ratio_resistance_3["ratio_resistance_3"] = ratio_resistance_3["high_touch"] / ratio_resistance_3["evaluated_candles"].replace(0, np.nan)
ratio_resistance_3 = ratio_resistance_3.sort_values("ratio_resistance_3", ascending=False).set_index("ema")

ratio_resistance_3


,high_touch,cross,evaluated_candles,ratio_resistance_3
ema,,,,
7,426,5303,8634,0.049340
6,405,5915,8635,0.046902
8,399,4847,8633,0.046218
11,396,3919,8630,0.045886
9,393,4508,8632,0.045528
...,...,...,...,...
170,63,784,8471,0.007437
195,60,697,8446,0.007104
196,58,692,8445,0.006868


__*4. Bearishness: what's the trend bias around this EMA?*__


For every time price pierced the EMA (in either direction), how many bars sat entirely below it?

Captures EMAs that mark decisive, uninterrupted trend regimes. \
Bearishness amplified by stability: EMA acted as a ceiling and was rarely breached.

- High ratio when price spends a lot of time below with few crossings -> bearish regime.
- Rewards EMAs where price spent most time below and the EMA was rarely breached.

Example: (high_touch + below) / cross = 850 / 150 = 5.67 \
Meaning: For every pierce, ~5.7 bars sat entirely below —> strong bearish regime.


In [22]:
ratio_resistance_4 = result[["ema", "high_touch", "below", "cross"]].copy()
ratio_resistance_4["ratio_resistance_4"] = (ratio_resistance_4["high_touch"] + ratio_resistance_4["below"]) / ratio_resistance_4["cross"].replace(0, np.nan)
ratio_resistance_4 = ratio_resistance_4.sort_values("ratio_resistance_4", ascending=False).set_index("ema")

ratio_resistance_4


,high_touch,below,cross,ratio_resistance_4
ema,,,,
199,73,4463,674,6.729970
198,71,4460,679,6.673049
197,64,4463,688,6.579942
196,58,4461,692,6.530347
195,60,4455,697,6.477762
...,...,...,...,...
5,346,633,6615,0.147997
4,239,351,7396,0.079773
3,130,103,8136,0.028638


__*5. Trend filter: fraction of bars where the entire candle was below the EMA.*__


Captures EMAs acting as a ceiling in downtrends, where price often respects the level without rallying into touch distance.

- A regime indicator: high ratio means price spent the period mostly below the EMA.
- Rewards EMAs that acted as a ceiling during trending stretches where price never rallied close enough to register a touch.
- Useful in strong downtrends where pure touch-based ratios under-count respected levels.


In [23]:
ratio_resistance_5 = result[["ema", "high_touch", "below", "cross", "evaluated_candles"]].copy()
ratio_resistance_5["ratio_resistance_5"] = (ratio_resistance_5["high_touch"] + ratio_resistance_5["below"]) / ratio_resistance_5["evaluated_candles"].replace(0, np.nan)
ratio_resistance_5 = ratio_resistance_5.sort_values("ratio_resistance_5", ascending=False).set_index("ema")

ratio_resistance_5


,high_touch,below,cross,evaluated_candles,ratio_resistance_5
ema,,,,,
199,73,4463,674,8442,0.537313
198,71,4460,679,8443,0.536658
197,64,4463,688,8444,0.536120
196,58,4461,692,8445,0.535110
195,60,4455,697,8446,0.534573
...,...,...,...,...,...
5,346,633,6615,8636,0.113363
4,239,351,7396,8637,0.068311
3,130,103,8136,8638,0.026974


__Visualization: hold rate (resistance quality) by EMA period.__

Visualization of how often each EMA's resistance test held vs broke, plus the hold rate.

Lets you see raw test volume and resistance quality on one chart.

Result: a dual-axis chart
- stacked bars for held vs broken counts
- a red line for the hold rate

total bar height = total resistance tests for that EMA: out of all tests, this many held (orange) and this many broke (blue). \
broken = resistance_test - resistance_held \
hold rate = resistance_held / resistance_test


In [44]:
#Plot unfiltered EMAs
#viz = ratio_resistance_1.reset_index().sort_values("ema")

#Plot only EMAs that pass both cross-frequency and sample-size filters
viz = ratio_resistance_1_filtered.reset_index().sort_values("ema")

ratio = viz["ratio_resistance_1"]
broke = viz["resistance_test"] - viz["resistance_held"]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=viz["ema"], y=viz["resistance_held"], name="resistance held",
           marker_color="orange", opacity=0.75),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(x=viz["ema"], y=broke, name="resistance broke",
           marker_color="steelblue", opacity=0.45),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=viz["ema"], y=ratio, name="resistance rate",
               mode="lines", line=dict(color="red", width=2),
               hovertemplate="%{y:.1%}"),
    secondary_y=True,
)

fig.update_layout(
    title="Resistance held vs broken by EMA period — with hold rate",
    barmode="stack",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    hoverlabel=dict(namelength=-1),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_xaxes(title_text="EMA period", tickprefix="EMA ")
fig.update_yaxes(title_text="Count", secondary_y=False)
fig.update_yaxes(title_text="Resistance rate", secondary_y=True, color="red", tickformat=".0%")

fig.show()


Use it for:
- Where the ratio peaks (the red line reaches its highest points on the chart) across the EMA spectrum —> EMAs periods that acted as the best resistance.
- Are they isolated spikes or broad hills (a whole zone respects resistance)?

### Universal EMA

EMA as any kind of S/R.

Three metrics are reported below:
- Hold rate — quality across both directions.
- Bounce rate — stricter restraint: fraction of interactions that didn't pierce.
- Tradability — frequency of universal S/R setup.

Flow: best metric → strict alternative → frequency check.

Which formula to use?

| # | formula | meaning | what it measures | usage |
|---|---|---|---|---|
| 1 | (support_held + resistance_held) / (support_test + resistance_test) | hold rate: of all S/R tests, what fraction held | When this EMA is tested from either side, how often does the close finish back on the supported side? | universal-quality metric |
| 2 | any_touch / (any_touch + cross) | bounce rate: fraction of interactions that didn't pierce | Does this EMA act as a wall, regardless of direction? How clean is this EMA's S/R (pure wick rejections, any pierce is a failure even if it recovered)? Of all interactions with the EMA (touches or crosses), what fraction did not pierce through? | stricter bounding metric |
| 3 | any_touch / evaluated_candles | frequency of clean S/R touches | Which EMA produces the most tradeable S/R setups in this dataset? How often does price reach this EMA at all? | tradability, activity gauge |


__*1. Hold rate: is this EMA a good universal S/R level to trade?*__


Universal quality metric: weighted average of the support and resistance hold rates.

ratio_universal_1 shows the % of S/R tests (in either direction) that held:
- 0.0 → every test broke through
- 0.5 → bounces half the time when tested
- 1.0 → every test held

Treats every cross as both a support test (close > EMA → support held) and a resistance test (close < EMA → resistance held). \
Each cross contributes 0.5 to the ratio (one direction held, the other broke), while pure touches (low_touch or high_touch) contribute 1.0.

Question: when price tests this EMA, how often does it get pushed back?


In [ ]:
# Universal quality rate (rejection rate, both directions): of all S/R tests on the EMA,
# what fraction held in either direction? Treats every cross as both a support and resistance test;
# resolves crosses by close (support held if close > EMA, resistance held if close < EMA) and aggregates.

ratio_universal_1 = result[["ema", "support_test", "support_held",
                            "resistance_test", "resistance_held",
                            "cross", "evaluated_candles"]].copy()
ratio_universal_1["ratio_universal_1"] = (
    (ratio_universal_1["support_held"] + ratio_universal_1["resistance_held"])
    / (ratio_universal_1["support_test"] + ratio_universal_1["resistance_test"]).replace(0, np.nan)
)
ratio_universal_1 = ratio_universal_1.sort_values("ratio_universal_1", ascending=False).set_index("ema")

ratio_universal_1.head(10)

,support_test,support_held,resistance_test,resistance_held,cross,evaluated_candles,ratio_universal_1
ema,,,,,,,
77,669,413,705,447,1136,8564,0.625910
84,652,398,669,421,1079,8557,0.619985
74,677,411,724,456,1153,8567,0.618844
78,666,406,694,435,1133,8563,0.618382
76,666,402,711,449,1140,8565,0.618010
85,645,393,668,417,1064,8556,0.616908
71,706,426,743,467,1180,8570,0.616287
79,666,404,686,428,1124,8562,0.615385
75,670,403,716,449,1149,8566,0.614719


__Filtering out the results__


In [46]:
# Top 10 EMAs that pass both filters: cross-frequency (structural) and sample-size (statistical)
# Sample size = total S/R tests across both directions.

ratio_universal_1_filtered = filter_by_cross_rate(ratio_universal_1)
total_test = ratio_universal_1_filtered["support_test"] + ratio_universal_1_filtered["resistance_test"]
ratio_universal_1_filtered = ratio_universal_1_filtered[total_test >= MIN_TOUCHES_UNI]
ratio_universal_1_filtered.head(10)

,support_test,support_held,resistance_test,resistance_held,cross,evaluated_candles,ratio_universal_1,cross_rate
ema,,,,,,,,
77,669,413,705,447,1136,8564,0.625910,0.132648
84,652,398,669,421,1079,8557,0.619985,0.126096
74,677,411,724,456,1153,8567,0.618844,0.134586
78,666,406,694,435,1133,8563,0.618382,0.132313
76,666,402,711,449,1140,8565,0.618010,0.133100
85,645,393,668,417,1064,8556,0.616908,0.124357
71,706,426,743,467,1180,8570,0.616287,0.137690
79,666,404,686,428,1124,8562,0.615385,0.131278
75,670,403,716,449,1149,8566,0.614719,0.134135


__*2. Bounce rate: does this EMA act as a wall?*__


Strict restraint metric: of all interactions with the EMA (touches or crosses), what fraction did not pierce through?

ratio_universal_2 shows the fraction of interactions where the EMA contained the bar's body:
- 0.0 → every interaction was a pierce
- 0.5 → half the interactions were wick-only
- 1.0 → no pierces (perfect wall)

Doesn't depend on close direction — only on whether the bar's range straddled the EMA. \
EMAs that get touched but not pierced score high regardless of trend.

Question: does the EMA act as a wall (regardless of direction)?


In [47]:
ratio_universal_2 = result[["ema", "any_touch", "cross", "evaluated_candles"]].copy()
ratio_universal_2["ratio_universal_2"] = ratio_universal_2["any_touch"] / (ratio_universal_2["any_touch"] + ratio_universal_2["cross"]).replace(0, np.nan)
ratio_universal_2 = ratio_universal_2.sort_values("ratio_universal_2", ascending=False).set_index("ema")

ratio_universal_2.head(10)


,any_touch,cross,evaluated_candles,ratio_universal_2
ema,,,,
53,349,1430,8588,0.196178
100,236,977,8541,0.194559
55,336,1392,8586,0.194444
52,343,1462,8589,0.190028
85,249,1064,8556,0.189642
101,227,971,8540,0.189482
115,212,913,8526,0.188444
98,230,991,8543,0.188370
143,192,828,8498,0.188235


__Filtering out the results__


In [48]:
# Sample size = bounce rate's denominator (any_touch + cross), not any_touch alone.

ratio_universal_2_filtered = filter_by_cross_rate(ratio_universal_2)
ratio_universal_2_filtered = ratio_universal_2_filtered[
    (ratio_universal_2_filtered["any_touch"] + ratio_universal_2_filtered["cross"]) >= MIN_TOUCHES_UNI
]
ratio_universal_2_filtered.head(10)


,any_touch,cross,evaluated_candles,ratio_universal_2,cross_rate
ema,,,,,
53,349,1430,8588,0.196178,0.166511
100,236,977,8541,0.194559,0.114389
55,336,1392,8586,0.194444,0.162124
52,343,1462,8589,0.190028,0.170218
85,249,1064,8556,0.189642,0.124357
101,227,971,8540,0.189482,0.113700
115,212,913,8526,0.188444,0.107084
98,230,991,8543,0.188370,0.116001
143,192,828,8498,0.188235,0.097435


__*3. Tradability: how often does price reach this EMA?*__


Evaluates how often the EMA produces any S/R setup (touch from above or below), not how good those setups are.

ratio_universal_3 shows the fraction of bars that registered as a clean S/R touch (no piercing).

- High ratio → frequently tested EMA, lots of opportunities
- Low ratio → rarely tested EMA (slow EMAs often live here)

Useful as an activity gauge: an EMA with high quality (ratio_universal_1) but very low tradability rarely fires.


In [ ]:
ratio_universal_3 = result[["ema", "any_touch", "cross", "evaluated_candles"]].copy()
ratio_universal_3["ratio_universal_3"] = ratio_universal_3["any_touch"] / ratio_universal_3["evaluated_candles"].replace(0, np.nan)
ratio_universal_3 = ratio_universal_3.sort_values("ratio_universal_3", ascending=False).set_index("ema")

ratio_universal_3

,any_touch,cross,evaluated_candles,ratio_universal_3
ema,,,,
7,859,5303,8634,0.099490
8,814,4847,8633,0.094289
10,794,4168,8631,0.091994
9,792,4508,8632,0.091752
11,784,3919,8630,0.090846
...,...,...,...,...
196,122,692,8445,0.014446
195,121,697,8446,0.014326
194,121,704,8447,0.014325


__Visualization: hold rate (universal quality) by EMA period.__


Visualization of how often each EMA's S/R test held vs broke (across both directions), plus the universal hold rate.

Lets you see raw test volume and universal quality on one chart.

Result: a dual-axis chart
- stacked bars for held vs broken counts
- a red line for the universal hold rate

total bar height = total S/R tests for that EMA: out of all tests, this many held (orange) and this many broke (blue). \
broken = (support_test + resistance_test) - (support_held + resistance_held) \
hold rate = (support_held + resistance_held) / (support_test + resistance_test)


In [51]:
#Plot unfiltered EMAs
#viz = ratio_universal_1.reset_index().sort_values("ema")

#Plot only EMAs that pass both cross-frequency and sample-size filters
viz = ratio_universal_1_filtered.reset_index().sort_values("ema")

held = viz["support_held"] + viz["resistance_held"]
total = viz["support_test"] + viz["resistance_test"]
broke = total - held
ratio = viz["ratio_universal_1"]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=viz["ema"], y=held, name="held (support + resistance)",
           marker_color="orange", opacity=0.75),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(x=viz["ema"], y=broke, name="broken",
           marker_color="steelblue", opacity=0.45),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=viz["ema"], y=ratio, name="universal hold rate",
               mode="lines", line=dict(color="red", width=2),
               hovertemplate="%{y:.1%}"),
    secondary_y=True,
)

fig.update_layout(
    title="Universal quality by EMA period — held vs broken with hold rate",
    barmode="stack",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    hoverlabel=dict(namelength=-1),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_xaxes(title_text="EMA period", tickprefix="EMA ")
fig.update_yaxes(title_text="Count", secondary_y=False)
fig.update_yaxes(title_text="Universal hold rate", secondary_y=True, color="red", tickformat=".0%")

fig.show()


What you see on the chart:
- **Stacked bars** show the total number of S/R tests for each EMA, broken into held (orange) and broken (blue). \
  Tall bars = heavily tested EMAs; short bars = rarely tested (treat ratios there with caution).
- **Red line** is the universal hold rate: of all S/R tests on this EMA, the fraction that held in either direction (close finished on the supported side).
- **Fast EMAs** (left side, after the cross-saturation filter) carry meaningful test counts but rarely cleanly hold — the line tends to sit lower.
- **Medium EMAs** (middle) are the sweet spot: enough tests to be statistically meaningful, with the hold rate often peaking here.
- **Slow EMAs** (right side) shrink in absolute count — the bars get shorter — and the hold rate becomes noisy.

Cross-reference with support_test + resistance_test before trusting a high ratio. \
The red line's peak (over a sample size you trust) marks the best universal S/R candidates.


## Visual analysis

An interactive financial chart in Plotly:
- Overlays EMAs on a candlestick chart to see if they line up with swing highs and lows.
- Chart opens showing the last 3 days.
- Drag the range slider or click 1d/1w/1m/3m buttons to jump to different windows.
- After jumping, click the "Autoscale" button in the top-right toolbar to rescale y-axis. Double-clicking the y-axis also works.

Change pd.Timedelta(days=3) if you want a different default window:
- days=1 for tighter zoom
- days=7 for a weekly view

__Configurable parameters:__

In [54]:
symbol                                                    # label for the candlestick trace and title, configured at data loading
ema_periods     = [53, 77, 84, 114]                       # a list with any number of EMA periods
ema_colors      = ["green", "orange", "blue", "purple"]   # one color per period
initial_window  = pd.Timedelta(days=3)                    # default zoom: how much history to show on load
chart_height    = 900

In [56]:
# Compute EMAs dynamically
for period in ema_periods:
    df[f"ema_{period}"] = df["close"].ewm(span=period, adjust=False).mean()

fig = go.Figure()

# Price candles
fig.add_trace(go.Candlestick(
    x=df["timestamp"],
    open=df["open"], high=df["high"], low=df["low"], close=df["close"],
    name=symbol,
))

# EMA lines
for period, color in zip(ema_periods, ema_colors):
    fig.add_trace(go.Scattergl(
        x=df["timestamp"],
        y=df[f"ema_{period}"],
        line=dict(color=color, width=1.5),
        name=f"EMA {period}",
    ))

# Default view: last N days
initial_start = df["timestamp"].iloc[-1] - initial_window
initial_end   = df["timestamp"].iloc[-1]

# Title built from config
ema_label = " & ".join(f"EMA {p}" for p in ema_periods)
chart_title = f"{symbol} — {ema_label}"

fig.update_layout(
    title=chart_title,
    height=chart_height,
    template="plotly_white",
    xaxis=dict(
        range=[initial_start, initial_end],
        rangeslider=dict(visible=True, thickness=0.04),
        rangeselector=dict(
            buttons=[
                dict(count=1, label="1d", step="day",   stepmode="backward"),
                dict(count=7, label="1w", step="day",   stepmode="backward"),
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=3, label="3m", step="month", stepmode="backward"),
                dict(step="all", label="All"),
            ]
        ),
    ),
    yaxis=dict(autorange=True, fixedrange=False),
)

fig.show()

## Backtesting

Translates the EMA touch analysis into a fully simulated trading strategy with realistic execution (fees + slippage), configurable stop loss / take profit / position sizing, and per-trade metrics including expectancy, profit factor, max drawdown, and exit-reason breakdown.

__Strategy logic:__
- Entry timing: at the close of the touch candle, with slippage applied.
- Long Entry: a candle's Low touches a chosen support EMA (within delta) and the candle closes back at/above the EMA (rejection). \
  Optionally gated by close > EMA(regime_filter).
- Short entry: a candle's High touches a chosen resistance EMA (within delta) and the candle closes back at/below the EMA (rejection). \
  Optionally gated by close < EMA(regime_filter).
- One position at a time.
- Exit: stop loss, take profit, or end-of-data — whichever fires first. \
  If both SL and TP are reachable in the same candle, SL is assumed to fire first (conservative).

__Optional regime_filter:__
- adds a broader trend check
- it is a second, optional entry condition that has to also be true for an entry to fire

The default entry signal is local: it tells you about the immediate behavior around the entry EMA, but says nothing about the broader trend. \
If you set regime_filter=200, the engine computes a second, slower EMA (e.g. EMA-200) and only allows a long entry when the touch candle's close is above it. \
What it means: "I'm only willing to buy a pullback if the bigger picture is also bullish (price above EMA-200)." \
Mirror logic for shorts: close < EMA(regime_filter) — you only short rallies when the broader trend is also down.

*Example:* \
BTC is in a long downtrend on the daily but pulled back UP to a fast 20-EMA on the 15-min and bounced. \
Without a regime filter, the strategy buys that bounce — and gets crushed when the daily downtrend resumes. \
With regime_filter=200, the engine refuses to buy because close < EMA-200 (the macro trend is down), and you avoid the trap.

__Configurable knobs:__
- direction: long / short / both
- stop_loss.mode: fixed_value | fixed_pct | beyond_ema | trailing | chandelier
- take_profit.mode: fixed_value | fixed_pct | r_multiple | trailing
- position_sizing.mode: fixed_value (notional in quote ccy) | fixed_pct (% of equity)
- fee_pct: per-fill, in % (e.g. 0.055 for ByBit taker)
- slippage_pct: per-fill, in % of price
- regime_filter: none, longs only when close > EMA(N), shorts only when close < EMA(N)

### 1. Setup

Single import + autoreload so edits to backtest.py hot-reload without restarting the kernel.

In [ ]:
# Engine + visualisation lives in backtest.py — import once and call from the cells below.
# %autoreload picks up edits to backtest.py without restarting the kernel.
# You only need to run them once per kernel session. Subsequent edits to backtest.py will pick up automatically without re-importing or restarting.

%load_ext autoreload
%autoreload 2

from dataclasses import replace
from backtest import (
    StopLossConfig, TakeProfitConfig, PositionSizingConfig, StrategyConfig,
    backtest, compute_metrics, print_metrics,
    plot_equity_curve, plot_trades_on_chart,
    walk_forward_split,
    sweep_configs, sweep_grid,
)

### 2. Helpers used here

These two pickers stay in the notebook (rather than backtest.py) because they consume the analyze_ema_touches output defined earlier.

In [ ]:
def pick_best_ema_support(train_df: pd.DataFrame, ema_range: Iterable[int],
                          delta: float, delta_mode: str,
                          min_touches: int = 30) -> int:
    """Return the EMA period with the highest support ratio on the train slice."""
    res = analyze_ema_touches(train_df, ema_range, delta, delta_mode)
    res = res[res["low_touch"] >= min_touches].copy()
    if len(res) == 0:
        raise ValueError(f"No EMA met min_touches={min_touches} on train slice")
    res["ratio_support_1"] = res["low_touch"] / (res["low_touch"] + res["cross"]).replace(0, np.nan)
    best = res.sort_values("ratio_support_1", ascending=False).iloc[0]
    return int(best["ema"])


def pick_best_ema_resistance(train_df: pd.DataFrame, ema_range: Iterable[int],
                             delta: float, delta_mode: str,
                             min_touches: int = 30) -> int:
    """Return the EMA period with the highest resistance ratio on the train slice."""
    res = analyze_ema_touches(train_df, ema_range, delta, delta_mode)
    res = res[res["high_touch"] >= min_touches].copy()
    if len(res) == 0:
        raise ValueError(f"No EMA met min_touches={min_touches} on train slice")
    res["ratio_resistance"] = res["high_touch"] / (res["high_touch"] + res["cross"]).replace(0, np.nan)
    best = res.sort_values("ratio_resistance", ascending=False).iloc[0]
    return int(best["ema"])


### 3. Capital, sizing, fees, slippage

__Sizing & capital — what to set, and why it matters__

The position sizer turns a notional dollar amount into a coin quantity. ByBit's BTCUSDT minimum is __0.001 BTC__, so any trade whose computed size falls below that gets skipped (the engine prints a warning the first time and a count at the end of the run).

To clear the floor, your per-trade notional must be ≥ min_size × price. At BTC ≈ &#36;70k that's ≈ &#36;70 per trade; at &#36;100k it's ≈ &#36;100. So:

- position_sizing=fixed_pct(10) on initial_capital=10_000 → &#36;1000/trade. Comfortable across all BTC price levels and through deep drawdowns.
- position_sizing=fixed_pct(10) on initial_capital=1_000 → &#36;100/trade. Works at BTC ≤ ≈ &#36;100k, but a >30% drawdown shrinks notional below the floor and trades start getting silently skipped.
- position_sizing=fixed_pct(10) on initial_capital=100 → &#36;10/trade. Always below the floor at any plausible BTC price → __zero trades__.
- position_sizing=fixed_value(100) → flat &#36;100/trade regardless of equity. Cleanest way to test on small capital; tradeoff is no compounding on the upside and no automatic de-risking on the downside.

Friction matters more on small capital. ByBit taker fee 0.055% × 2 fills + slippage = ≈ 0.13% per round-trip. On a &#36;100 notional that's ≈ &#36;0.13/trade — same in absolute terms regardless of equity, but a larger fraction of a smaller account.

__Rule of thumb:__ pick capital and sizing such that even a 30–50% drawdown still produces notional ≥ min_size × max_expected_price. If you want to backtest realistically on a small account, use fixed_value sizing.


### 4. Walk-forward split + EMA selection

70/30 split — fit on the first 70% (train), evaluate on the held-out 30% (test). \
Pick the best EMA on train only via ratio_support_1 (longs) or ratio_resistance (shorts). \
Anything refitted on test is overfit by definition.


In [ ]:
# Reuse df already fetched above.
# Default config: longs only, 1% fixed stop loss, 3R take-profit, 10% of equity per trade, 0.055% ByBit taker fee , 0.01% slippage. 
# EMA chosen via support ratio on a train slice.

train_df, test_df = walk_forward_split(df, train_pct=0.7)
print(f"Train: {train_df['timestamp'].iloc[0]} → {train_df['timestamp'].iloc[-1]}  ({len(train_df)} candles)")
print(f"Test:  {test_df['timestamp'].iloc[0]} → {test_df['timestamp'].iloc[-1]}  ({len(test_df)} candles)")

best_ema = pick_best_ema_support(train_df, ema_range=range(15, 200), delta=delta, delta_mode=delta_mode, min_touches=20)
print(f"Best support EMA on train: {best_ema}")


### 5. Strategy configuration

Assemble the StrategyConfig from the sizing / fee choices above and the EMA picked on train.

In [ ]:
cfg = StrategyConfig(
    direction="long",
    ema_period=best_ema,
    delta=delta,
    delta_mode=delta_mode,
    stop_loss=StopLossConfig(mode="fixed_pct", value=1.0),       # 1% stop
    take_profit=TakeProfitConfig(mode="r_multiple", value=3.0),  # 3R target
    position_sizing=PositionSizingConfig(mode="fixed_value", value=100.0),  # fixed_pct is % of equity per trade
    fee_pct=0.055,        # ByBit taker
    slippage_pct=0.01,    # 0.01% of price
    initial_capital=100.0,
)
cfg


### 6. Run on train (in-sample)

Fit the strategy to the training slice. The numbers below will look optimistic — that's expected; the test slice (next section) is the honest read.

In [ ]:
trades_train, equity_train = backtest(train_df, cfg)
metrics_train = compute_metrics(trades_train, equity_train, cfg.initial_capital)
print_metrics(metrics_train)

Equity curve and drawdown over time.

In [ ]:
plot_equity_curve(equity_train, cfg.initial_capital, title=f"Equity curve — train (EMA {cfg.ema_period})")

Trades on the price chart — blue ▲ longs, red ▼ shorts, ✕ exits color-coded by reason.

In [ ]:
plot_trades_on_chart(train_df, trades_train, cfg, initial_window_days=30)

### 7. Run on test (out-of-sample) — the honest number

Same config on data the EMA picker never saw. If train and test agree (both positive after fees), there's a candidate edge. If test collapses while train looked great, the EMA selection over-fit.

In [ ]:
trades_test, equity_test = backtest(test_df, cfg)
metrics_test = compute_metrics(trades_test, equity_test, cfg.initial_capital)
print_metrics(metrics_test)

Equity curve and drawdown over time.

In [ ]:
plot_equity_curve(equity_test, cfg.initial_capital, title=f"Equity curve — test (EMA {cfg.ema_period})")

Trades on the price chart — blue ▲ longs, red ▼ shorts, ✕ exits color-coded by reason.

In [ ]:
plot_trades_on_chart(test_df, trades_test, cfg, initial_window_days=30)

### 8. Reading the result

- If __test expectancy > 0 after fees__ and is in the same ballpark as train → there's a real edge worth paper-trading next.
- If __test expectancy < 0__ while train was positive → the EMA selection over-fit the train slice. Try a coarser ema_range, a longer history, or different delta.
- If __profit factor < 1.2__ on test → edge is too thin to survive real-world frictions you didn't model (funding, partial fills, exchange downtime).
- __Exit reason mix__ tells you whether the 3R target is realistic. If stop_loss >> take_profit, you're getting stopped out before the move develops — try wider stops, lower R-targets, or a slower EMA.

__Next iterations__ (separate experiments, not this run):
1. Sweep EMA period × stop% × R-target on train, pick top 3 by Sortino, validate each on test.
2. Sweep direction (long / short / both). In a chop-heavy regime both typically beats either side alone because mean-reversion fires on both swings; in strong trends, the trending direction wins.
3. Add a regime filter (e.g. only trade longs when price is above a slow EMA like 200) to skip counter-trend stretches.
4. Replace single-asset with a basket (BTC + ETH + a couple of liquid alts) — diversifies the edge.

Train and test agree → not curve-fit, just a non-edge as configured. \
The 3R target only fires 22% of the time, and 1% stops absorb the rest.

Variations to sweep next: 
- wider stops (1.5–2%), 
- lower R-targets (1.5–2R), 
- slower EMAs (100+),
- add a regime filter (only trade longs while close > EMA-200).

__Metrics:__

__*Expectancy:*__
- = win_rate * avg_win + (1 - win_rate) * avg_loss
- Folds win size and loss size into one number.
- Is sensitive to trade frequency.
- Meaning: the expected net P&L per trade in quote currency (USDT), after fees and slippage, weighted by win/loss probability.
- Question: if I take one more trade with this configuration, what's my expected dollar outcome?
- Example: expectancy = −2.52 means: across N trades the strategy loses $2.52 of net P&L per trade on average.
- Results:
    - Positive = the strategy makes money on average per trade.
    - Negative = the strategy loses money on average.

__*Profit factor:*__
- = (gross_win / gross_loss) if gross_loss > 0 else float("inf")
- Expectancy expressed as a ratio: PF > 1 ⟺ expectancy > 0.
- Is not sensitive to trade frequency; tells the edge per dollar at risk regardless of how often you trade.
- Meaning: the ratio of cumulative wins to cumulative losses; ratio of total dollars won to total dollars lost across all trades.
- Question: For every $1 the strategy lost, how many dollars did it win back?
- Results:
    - < 1.0 unprofitable
    - 1.0–1.3 marginal, easily eaten by a regime change or fee bumps. PF = 0.50 means you win $0.50 for every $1 lost.
    - 1.3–1.7 decent, tradeable
    - 1.7–3.0 strong. PF = 2.0 means you win $2 for every $1 lost.
    - .> 3.0 suspiciously good — usually overfitting or a small sample

__*Max drawdown %:*__
- = (equity_curve - equity_curve.cummax()) / equity_curve.cummax() * 100.min()
- Captures the worst point of the worst losing streak.
- Reflects how sustainable a strategy is in terms of risk of ruin. 
- Two strategies can have identical final P&L but very different max DDs. In live trading, positions are often closed during a 33% drawdown before any recovery occurs. 
- Meaning: the deepest peak-to-trough decline of the equity curve, expressed as a percent of the prior peak.
- Question: What's the worst paper loss I'd have lived through to keep running this strategy?
- Results:
    - A common practice is to assume the live max DD will be 1.3–2× the backtested max DD, especially for strategies with negative or low expectancy.
    - It's a single point in time — doesn't tell you how long the drawdown lasted or how many times you suffered a similar one.

__*Sharpe:*__
- = (returns.mean() / returns.std() * np.sqrt(252)) if returns.std() > 1e-4 else 0.0
- Is used here for ranking configs against each other within this codebase. 
- Is built  around daily per-trade returns, not annualized -> is not valid for comparing with industry benchmarks.
- Meaning: risk-adjusted return. How much return did you earn per unit of risk? 
- Question: Per unit of overall return variability, how much edge?

__*Sortino:*__
- = (returns.mean() / downside.std() * np.sqrt(252)) if len(downside) > 1 and downside.std() > 1e-4 else 0.0, with downside = returns[returns < 0]
- Is used here for ranking configs against each other within this codebase.
- Is built  around daily per-trade returns, not annualized -> is not valid for comparing with industry benchmarks.
- Meaning: risk-adjusted return. How much return did you earn per unit of risk? 
- Question: Per unit of downside pain, how much edge?
- Results: 
    - Sharpe ≈ Sortino → wins and losses have similar size distributions (typical for fixed-stop / fixed-target strategies).
    - Sortino > Sharpe → upside is fatter-tailed than downside (classic for trend-following: many small losses, occasional big winners).
    - Sortino < Sharpe → downside is fatter-tailed (mean-reversion sold without a stop, vol selling — the "picking up nickels in front of a steamroller" profile).

### 9. Parameter sweep — stop% × R-target

Run a small grid over (stop%, R-target) on the **same train/test split**, comparing the baseline (1% / 3R) against the variations: wider stop (1.5%) and lower target (2R).

Why grid these two together: a wider stop and a smaller R-target both push toward higher win-rate, lower-magnitude trades. We want to see whether either tweak — or the combination — turns expectancy positive after fees.

sweep_configs(df, base_cfg, stop_pcts, r_targets) runs the full grid and returns a tidy DataFrame ranked by expectancy, with exit-reason counts inline (sl / tp / trail / eod).

__Run the grid on train and test__

In [ ]:
stop_pcts = [1.0, 1.5]
r_targets = [2.0, 3.0]

sweep_train = sweep_configs(train_df, cfg, stop_pcts, r_targets)
sweep_test  = sweep_configs(test_df,  cfg, stop_pcts, r_targets)

print("=== TRAIN ===")
print(sweep_train.to_string(index=False))
print("\n=== TEST ===")
print(sweep_test.to_string(index=False))


__How to read the table:__

- Each row is one full backtest with a different (stop_pct, r_target). Rows are sorted by expectancy (highest first).
- sl/tp/trail/eod are exit-reason counts — use them to sanity-check that more trades are reaching TP at lower R-targets and wider stops, as expected.
- The decision is: **does any row have positive expectancy on the test slice, and is its train rank similar?** If train ranks A > B > C > D and test ranks roughly agree, the ordering is real signal. If test re-shuffles randomly, the differences are noise.

__Result:__

The 1.5%/3R config was the worst on train (rank 4) and best on test (rank 1, the only positive). \
That's the opposite of the train→test agreement you want. \
With only 15 trades on test, this is almost certainly noise rather than signal. \
A truly robust edge would rank similarly on both slices.

__What this actually tells you:__

1. None of the four configs has a real edge in this 3-month window.
2. The fact that a bad-on-train config "won" on test is exactly why walk-forward exists — it stops you from deploying the 1.5%/3R variant just because the test number looks good.
3. To find a real edge, expand the grid: more EMA periods, slower timeframes, sweep direction (long / short / both), get more data (12+ months), or change the entry signal entirely (the touch+rejection logic may not be enough).

In [ ]:
best_test = sweep_test.iloc[0]
print(f"Best on test:  stop={best_test['stop_pct']}%   target={best_test['r_target']}R")
print(f"  expectancy={best_test['expectancy']}   profit_factor={best_test['profit_factor']}   return={best_test['total_return_pct']}%")

from dataclasses import replace
best_cfg = replace(
    cfg,
    stop_loss=StopLossConfig(mode="fixed_pct", value=float(best_test["stop_pct"])),
    take_profit=TakeProfitConfig(mode="r_multiple", value=float(best_test["r_target"])),
)
trades_best, equity_best = backtest(test_df, best_cfg)
print()
print_metrics(compute_metrics(trades_best, equity_best, best_cfg.initial_capital))


In [ ]:
plot_equity_curve(equity_best, best_cfg.initial_capital,
                  title=f"Equity — test, stop={best_cfg.stop_loss.value}% / target={best_cfg.take_profit.value}R")


In [ ]:
plot_trades_on_chart(test_df, trades_best, best_cfg, initial_window_days=30)

### 10. Full sweep — stop% × R-target × EMA × regime_filter × direction

Extend the grid to up to five axes: every combination of (stop_pct, r_target, ema_period, regime_filter, direction).

- __EMA period__: include slower EMAs (50, 100, 150, 200) — the "respected" levels traders actually watch.
- __Regime filter__: None = no filter; 50 = only enter longs while close > EMA-50 (or only shorts while close < EMA-50).
  - Caveat: when the regime EMA equals or is faster than the entry EMA, the filter becomes redundant or near-redundant. The interesting cases are entry EMA ≥ regime EMA (e.g. entry on EMA-100 retrace, gated by EMA-200 regime).
- __Direction__: "long", "short", or "both". Sweeping this lets the data tell you whether the period is dominated by uptrend pullbacks, downtrend bounces, or balanced mean-reversion.
- The grid in the example below is 2 × 2 × 4 × 2 = __32 backtests__ per slice. Adding direction would push it to 96. All run in seconds on a few thousand candles.

We refit nothing on the test slice; we just rank the same configs on both train and test and look for configs whose ranks agree.

__Run the 32-combo grid__

Keep regime_filter ≥ max(ema_period) (or None), otherwise you get NaN rows when a combo has regime_filter < ema_period.

The same rule holds for shorts: a regime EMA much faster than the entry EMA blocks almost all entries. In a downtrend, price pulling UP to a slow entry EMA is by definition above any faster EMA, so close < regime_EMA fails — and shorts never fire.

In [ ]:
grid = {
    "stop_pct":       [0.5, 1.0, 1.5],
    "r_target":       [2.0, 2.5, 3.0],
    "ema_period":     [20, 50, 54, 115, 118],
    "regime_filter":  [None, 200],
}

sweep4_train = sweep_grid(train_df, cfg, grid)
sweep4_test  = sweep_grid(test_df,  cfg, grid)

print("=== TRAIN — top 10 by expectancy ===")
print(sweep4_train.head(10).to_string(index=False))
print("\n=== TEST — top 10 by expectancy ===")
print(sweep4_test.head(10).to_string(index=False))


__Result:__

Train → test agreement on the same config (stop 1.5% / 3R / EMA-200 / regime EMA-50). \
This is the first config across the whole exploration where the train pick is also positive on test. \
The shape of the edge: very slow EMA (200) + macro filter (only buy when close > EMA-50) + wider stop (1.5%) + asymmetric target (3R). \
Effectively "buy retracements to a major dynamic level only when short-term momentum is up."

__Caveat:__ 

18 train + 8 test trades is tiny — a few extra wins or losses would flip the ranking. Not for trade yet.

Further actions:

1. Re-run on 12+ months of data (current window is just 90 days). With ~10× the trade count, the 1.5%/3R/EMA-200/regime-50 number will either solidify or fall apart.
2. Add adjacent EMA periods (175, 225, 250) — if the edge is real, neighboring EMAs should also rank well.
3. Test on ETH and a couple of other liquid majors — a true edge generalizes.

__Pick by train expectancy, judge on test__

The honest workflow: pick the best config *as ranked on train*, then look up that exact config's row on test. If the train-best is also profitable on test, you have a candidate edge. If it collapses on test, the apparent edge was overfit.

In [ ]:
# Best on TRAIN (the only slice you're allowed to pick from)
key_cols = ["stop_pct", "r_target", "ema_period", "regime_filter"]
train_pick = sweep4_train.iloc[0]
print("Train pick:")
print(train_pick[key_cols + ["trades", "win_rate", "expectancy", "profit_factor", "total_return_pct"]].to_string())

# Find that exact config on test
match = sweep4_test
for k in key_cols:
    v = train_pick[k]
    if pd.isna(v):
        match = match[match[k].isna()]
    else:
        match = match[match[k] == v]
print()
print("Same config on test:")
if len(match) == 0:
    print("  (no matching row — should not happen with shared grid)")
else:
    print(match.iloc[0][key_cols + ["trades", "win_rate", "expectancy", "profit_factor", "total_return_pct"]].to_string())


__Top-3 configs on test__ (for context — these are NOT picks, just the leaderboard)

In [ ]:
sweep4_test.head(3)[key_cols + ["trades", "win_rate", "expectancy", "profit_factor", "total_return_pct", "max_dd_pct", "sl_n", "tp_n"]]


__How to read these tables:__

- __Look for rank agreement, not point estimates.__ A config that's rank-2 on train and rank-4 on test is a stronger signal than one that's rank-25 on train and rank-1 on test (the latter is overfit-luck).
- __Check trades per row.__ With ~6000 train candles, a 50-EMA setup with regime filter might produce 30–60 trades; a 200-EMA with filter might produce <10. Anything with <20 trades is a noisy estimate.
- __Sample-size adjustment__: when comparing test rows, prefer configs with more trades — small trades makes expectancy unstable.
- __Regime filter check__: compare otherwise-identical rows with regime_filter=None vs a set value. The filter should reduce trade count and *might* improve win rate. If win rate doesn't move, the filter isn't doing anything useful at the chosen EMAs.
- __Direction balance (when sweeping direction='both')__: a healthy both run shows roughly comparable long_trades and short_trades counts on a chop-heavy slice; a lopsided count means one side is being filtered out by the regime/touch conditions for most of the period — check whether you'd have done better picking that side explicitly.

__If nothing positive on test:__ this dataset (3 months) probably doesn't contain enough variety. Re-run with 12+ months of history; the longer window will both stabilize statistics and let you walk forward across regime changes.

## Data export

Saving dataframes to CSV for further analysis in Excel or Tableau.

In [ ]:

filename_df = f"df_{symbol}_{interval}_{start}_{end}.csv"
df.to_csv(filename_df, index=False)
print(f"\nResults saved to {filename_df}")

In [ ]:
filename_ema = f"ema_analysis_{symbol}_{interval}_{start}_{end}_{ema_range}_{delta}.csv"
result.to_csv(filename_ema, index=False)
print(f"\nResults saved to {filename_ema}")

## Conclusion

__Position:__
- For a short position: use EMAs with the highest resistance ratio.
- For a long position: use EMAs with the highest support ratio.

__EMA choice:__
- The strongest EMA candidates should have the best mix of ratio and sample size.
- EMAs with slightly higher ratio but fewer touches can acts as a secondary/ slower level.

__EMA confirmation:__
- Overlay chosen EMAs on the interactive financial chart to see if they line up with swing highs and lows. \
If yes — you've found dynamic support & resistance levels that algorithmically survived price action during the set time period.

__Further actions:__
- To use the analysis results in TradingView: 
    - find suitable EMAs here
    - paste the EMA periods you found into a Pine indicator (e.g. ta.ema(close, 50))
    - plot these periods in Pine Script on the chart.